# 13 — Aim 2 D3 Curated ALT: Robustness Analysis

Four extensions to the notebook 12 D3 (curated ALT-only) analysis:

1. **1000-permutation test** — re-run D3 (primary + pragmatic) with 1000 label shuffles instead of 100 for better-resolved p-values.
2. **Known-sex-only pragmatic sensitivity** — drop the four Unknown-sex samples (ACH-001813, ACH-001817, ACH-002779, ACH-003573) and re-run D3 on N=27. Reports whether AUROC and top-coefficient genes are stable to that exclusion.
3. **Bootstrap coefficient stability** — 200 bootstrap resamples of the training set (with replacement), refit the curated 19-gene elastic net at the fold-median C, flag each gene as stable (non-zero >80% of bootstraps), borderline (50–80%), or unstable (<50%).
4. **LOOCV fold coefficient frequency** — for each LOOCV held-out-fold refit, record which of the 19 genes has a non-zero coefficient; report per-gene frequency across all N folds.

Idempotent, sha256-verified. Does not modify notebook 12 or `src/aim2_signature.py`.

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.utils import log_versions, sha256_file
import pandas as pd

log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'joblib'])
print(f'Run: {RUN_TIMESTAMP}')


ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline
Library versions:
  pandas: 3.0.2
  numpy: 2.4.4
  scipy: 1.17.1
  sklearn: 1.8.0
  joblib: 1.5.3
Run: 2026-06-12T18:02:13Z


## 1 — Imports and configuration

In [2]:
from src.aim2_signature import (
    apply_inclusion_filter, assert_expected_counts, assert_ground_truth,
    direction_counts, labels_from_direction, load_expression,
    evaluate, bootstrap_ci,
    ALT_GENE_LIST, C_GRID, TOP_K_VARIABLE_GENES,
    N_BOOTSTRAP, RANDOM_SEED, L1_RATIO, INNER_CV_FOLDS,
    strip_gene_symbol, build_estimator, LoocvResult, VarianceGeneFilter,
)
from src.aim1_inventory import DEPMAP_RNA_PATH, DEPMAP_RNA_DATA_TYPE, depmap_rna_model_ids
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, LeaveOneOut
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.preprocessing import StandardScaler
import json

MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

N_PERMUTATIONS_LONG = 1000  # vs 100 in nb12
N_BOOTSTRAP_COEF = 200      # bootstrap resamples for coefficient stability
NONZERO_THRESH = 1e-10      # |coef| > threshold → non-zero

# The four Unknown-sex samples in the pragmatic cohort
UNKNOWN_SEX_IDS = frozenset(['ACH-001813', 'ACH-001817', 'ACH-002779', 'ACH-003573'])

print(f'N_PERMUTATIONS_LONG={N_PERMUTATIONS_LONG}  '
      f'N_BOOTSTRAP_COEF={N_BOOTSTRAP_COEF}  '
      f'RANDOM_SEED={RANDOM_SEED}')
print(f'RNA matrix: {DEPMAP_RNA_PATH.relative_to(ROOT)}  [{DEPMAP_RNA_DATA_TYPE}]')


N_PERMUTATIONS_LONG=1000  N_BOOTSTRAP_COEF=200  RANDOM_SEED=42
RNA matrix: data/raw/depmap/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv  [log2(TPM+1) protein-coding]


## 2 — Load data and curated gene set

In [3]:
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)
depmap_models = depmap_rna_model_ids(DEPMAP_RNA_PATH)
DEFINITIONS = ['primary', 'pragmatic']

# Inclusion filters (same as nb11 / nb12)
filtered = {}
for defn in DEFINITIONS:
    df, _ = apply_inclusion_filter(manifest, depmap_models, defn)
    assert_expected_counts(df, defn)
    assert_ground_truth(df)
    filtered[defn] = df
    print(f'[{defn}] N={len(df)}  {direction_counts(df)}')

# Known-sex-only pragmatic: drop 4 Unknown-sex samples
prag_ks = filtered['pragmatic'][
    ~filtered['pragmatic']['sample_id'].isin(UNKNOWN_SEX_IDS)
].reset_index(drop=True)
print(f'\n[pragmatic known-sex] N={len(prag_ks)}  {direction_counts(prag_ks)}')
print(f'  Dropped: {sorted(UNKNOWN_SEX_IDS)}')

# Labels and sample IDs
y = {defn: labels_from_direction(filtered[defn]) for defn in DEFINITIONS}
ids = {defn: filtered[defn]['sample_id'].tolist() for defn in DEFINITIONS}
y_ks = labels_from_direction(prag_ks)
ids_ks = prag_ks['sample_id'].tolist()

# Load expression (pragmatic superset)
print('\nLoading expression ...')
t0 = time.time()
expr_all = load_expression(ids['pragmatic'])
gene_cols = list(expr_all.columns)
print(f'  {expr_all.shape[0]} x {expr_all.shape[1]}  ({time.time()-t0:.0f}s)')

# Curated ALT gene columns (19/21 in matrix)
gene_sym_to_col = {strip_gene_symbol(c): c for c in gene_cols}
found_curated = [(sym, gene_sym_to_col[sym])
                 for sym in ALT_GENE_LIST if sym in gene_sym_to_col]
not_found_curated = [sym for sym in ALT_GENE_LIST if sym not in gene_sym_to_col]
curated_cols = [col for _, col in found_curated]
curated_syms = [sym for sym, _ in found_curated]
curated_col_idx = [gene_cols.index(c) for c in curated_cols]
print(f'\nCurated genes: {len(curated_syms)}/{len(ALT_GENE_LIST)} found  '
      f'(not found: {not_found_curated})')

# Curated feature matrices
X = {defn: expr_all.loc[ids[defn]].to_numpy(dtype=float)[:, curated_col_idx]
     for defn in DEFINITIONS}
X_ks = expr_all.loc[ids_ks].to_numpy(dtype=float)[:, curated_col_idx]

for defn in DEFINITIONS:
    print(f'[{defn}] X_curated={X[defn].shape}  '
          f'pos={y[defn].sum()}  neg={(y[defn]==0).sum()}')
print(f'[pragmatic_ks] X_curated={X_ks.shape}  '
      f'pos={y_ks.sum()}  neg={(y_ks==0).sum()}')


[primary] N=20  {'positive': 12, 'negative': 8}
[pragmatic] N=31  {'positive': 18, 'negative': 13}

[pragmatic known-sex] N=27  {'positive': 17, 'negative': 10}
  Dropped: ['ACH-001813', 'ACH-001817', 'ACH-002779', 'ACH-003573']

Loading expression ...
  31 x 19215  (8s)

Curated genes: 19/21 found  (not found: ['TERC', 'MRE11A'])
[primary] X_curated=(20, 19)  pos=12  neg=8
[pragmatic] X_curated=(31, 19)  pos=18  neg=13
[pragmatic_ks] X_curated=(27, 19)  pos=17  neg=10


## 3 — Custom helpers

All helpers reuse `VarianceGeneFilter`, `build_estimator`, `LoocvResult`, `C_GRID`, and `INNER_CV_FOLDS` imported from `src.aim2_signature`. No logic is duplicated.

In [4]:
def _inner_cv_splits(n_pos, n_neg, random_state=RANDOM_SEED):
    n_splits = max(2, min(INNER_CV_FOLDS, n_pos, n_neg))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)


def run_loocv_no_filter_with_coefs(
        X, y, sample_ids, random_state=RANDOM_SEED, fixed_C=None):
    """Outer LOOCV (StandardScaler -> LR, no variance filter).

    Returns (LoocvResult, fold_coefs) where fold_coefs[i] is the coefficient
    vector from the fold that held out sample i.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    n, n_feat = len(y), X.shape[1]
    loo = LeaveOneOut()
    probs = np.empty(n, dtype=float)
    selected_Cs = []
    fold_coefs = np.zeros((n, n_feat), dtype=float)

    for train_idx, test_idx in loo.split(X):
        X_tr_raw, X_te_raw = X[train_idx], X[test_idx]
        if fixed_C is not None:
            best_C = float(fixed_C)
        else:
            n_pos_tr = int(y[train_idx].sum())
            n_neg_tr = int(len(train_idx) - n_pos_tr)
            inner_pipe = SKPipeline([
                ('scaler', StandardScaler()),
                ('clf', build_estimator(C=1.0, random_state=random_state)),
            ])
            cv = _inner_cv_splits(n_pos_tr, n_neg_tr, random_state)
            grid = GridSearchCV(inner_pipe, {'clf__C': list(C_GRID)},
                                scoring='roc_auc', cv=cv, n_jobs=1, refit=False)
            grid.fit(X_tr_raw, y[train_idx])
            best_C = float(grid.best_params_['clf__C'])
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr_raw)
        X_te = sc.transform(X_te_raw)
        clf = build_estimator(C=best_C, random_state=random_state)
        clf.fit(X_tr, y[train_idx])
        i = test_idx[0]
        probs[i] = clf.predict_proba(X_te)[:, 1][0]
        selected_Cs.append(best_C)
        fold_coefs[i] = clf.coef_.ravel()

    return LoocvResult(list(sample_ids), y, probs, selected_Cs, []), fold_coefs


def permutation_test_no_filter(
        X, y, observed_auroc, fixed_C,
        n_perm=N_PERMUTATIONS_LONG, seed=RANDOM_SEED, progress_every=100):
    """Fast permutation null for the curated-ALT (no-filter) pipeline.

    Precomputes fold-local StandardScaler matrices once (label-independent),
    then re-fits only the classifier per permutation. Mathematically equivalent
    to re-running the full fold pipeline each time.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    rng = np.random.default_rng(seed)

    # Precompute fold scalers (X-only, label-independent)
    loo = LeaveOneOut()
    fold_data = []
    for train_idx, test_idx in loo.split(X):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X[train_idx])
        X_te = sc.transform(X[test_idx])
        fold_data.append((train_idx, test_idx, X_tr, X_te))

    perm_aurocs = []
    t0 = time.time()
    for i in range(n_perm):
        y_perm = rng.permutation(y)
        probs_p = np.empty(len(y), dtype=float)
        for train_idx, test_idx, X_tr, X_te in fold_data:
            clf = build_estimator(C=fixed_C, random_state=RANDOM_SEED)
            clf.fit(X_tr, y_perm[train_idx])
            probs_p[test_idx[0]] = clf.predict_proba(X_te)[:, 1][0]
        if len(np.unique(y_perm)) >= 2:
            perm_aurocs.append(float(roc_auc_score(y_perm, probs_p)))
        if progress_every and ((i + 1) % progress_every == 0 or i + 1 == n_perm):
            print(f'    perm {i+1}/{n_perm}  ({time.time()-t0:.0f}s)', flush=True)

    perm_arr = np.asarray(perm_aurocs)
    return {
        'observed_auroc': float(observed_auroc),
        'perm_aurocs': perm_arr,
        'p_value': float(np.mean(perm_arr >= observed_auroc)),
        'n_perm': len(perm_arr),
        'permutation_fixed_C': float(fixed_C),
    }


def bootstrap_coef_stability(
        X, y, fixed_C, n_boot=N_BOOTSTRAP_COEF,
        seed=RANDOM_SEED, random_state=RANDOM_SEED):
    """Bootstrap coefficient stability.

    Draws n_boot bootstrap resamples (with replacement), refits the curated
    elastic net at fixed_C on each, and returns the per-gene non-zero frequency
    (fraction of valid resamples where |coef| > NONZERO_THRESH).

    Returns (freq_array, n_valid) where n_valid <= n_boot (degenerate
    single-class resamples are skipped).
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    n, n_feat = X.shape
    rng = np.random.default_rng(seed)
    nonzero_counts = np.zeros(n_feat, dtype=int)
    n_valid = 0

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        X_b, y_b = X[idx], y[idx]
        if len(np.unique(y_b)) < 2:
            continue
        sc = StandardScaler()
        X_b_s = sc.fit_transform(X_b)
        clf = build_estimator(C=fixed_C, random_state=random_state)
        clf.fit(X_b_s, y_b)
        nonzero_counts += (np.abs(clf.coef_.ravel()) > NONZERO_THRESH).astype(int)
        n_valid += 1

    freq = nonzero_counts / max(n_valid, 1)
    return freq, n_valid


def classify_stability(freq):
    """stable (>0.80), borderline (0.50-0.80), unstable (<0.50)."""
    return np.where(freq > 0.80, 'stable',
           np.where(freq >= 0.50, 'borderline', 'unstable'))


print('Helpers defined: run_loocv_no_filter_with_coefs, permutation_test_no_filter,')
print('                 bootstrap_coef_stability, classify_stability')


Helpers defined: run_loocv_no_filter_with_coefs, permutation_test_no_filter,
                 bootstrap_coef_stability, classify_stability


## 4 — Analysis 1 + 4: D3 LOOCV with fold coefficient tracking

Run D3 LOOCV (no variance filter) for primary and pragmatic, recording the coefficient vector from each held-out fold's refit. These fold coefficients feed Analysis 4 (fold frequency). The LOOCV result is then used for the 1000-permutation test.

In [5]:
loocv_res = {}   # defn -> LoocvResult
fold_coefs = {}  # defn -> ndarray (N, 19)

for defn in DEFINITIONS:
    n_samp = len(ids[defn])
    print(f'\n=== D3 LOOCV [{defn.upper()}]  N={n_samp} ===')
    t0 = time.time()
    res, fc = run_loocv_no_filter_with_coefs(X[defn], y[defn], ids[defn])
    elapsed = time.time() - t0
    fixed_C = float(np.median(res.selected_Cs))
    print(f'  {elapsed:.1f}s  AUROC={res.auroc:.4f}  '
          f'median_C={fixed_C:.4f}  '
          f'C_range=[{min(res.selected_Cs):.4f},{max(res.selected_Cs):.4f}]')
    loocv_res[defn] = res
    fold_coefs[defn] = fc

metrics = {defn: evaluate(loocv_res[defn].y_true, loocv_res[defn].y_prob)
           for defn in DEFINITIONS}
cis = {defn: bootstrap_ci(loocv_res[defn].y_true, loocv_res[defn].y_prob,
                          n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
       for defn in DEFINITIONS}

print('\n--- Metrics (recomputed from fold predictions) ---')
for defn in DEFINITIONS:
    m, ci = metrics[defn], cis[defn]
    print(f'[{defn}] AUROC={m["auroc"]:.4f} '
          f'[{ci["auroc"][0]:.4f},{ci["auroc"][1]:.4f}]  '
          f'AUPRC={m["auprc"]:.4f}  '
          f'sens@90spec={m["sensitivity_at_90_spec"]:.4f}')



=== D3 LOOCV [PRIMARY]  N=20 ===
  1.8s  AUROC=0.9479  median_C=0.3162  C_range=[0.1000,1.0000]

=== D3 LOOCV [PRAGMATIC]  N=31 ===
  3.1s  AUROC=0.8291  median_C=0.3162  C_range=[0.1000,31.6228]

--- Metrics (recomputed from fold predictions) ---
[primary] AUROC=0.9479 [0.8125,1.0000]  AUPRC=0.9755  sens@90spec=0.9167
[pragmatic] AUROC=0.8291 [0.6681,0.9625]  AUPRC=0.9166  sens@90spec=0.7778


## 5 — Analysis 1 continued: 1000-permutation test

Label-shuffling permutation null with `N_PERMUTATIONS_LONG=1000`. Fold-local StandardScaler preprocessing is precomputed once (label-independent); only the classifier is refit per permutation. Fixed C = median LOOCV-selected C.

In [6]:
perms = {}

for defn in DEFINITIONS:
    res = loocv_res[defn]
    fixed_C = float(np.median(res.selected_Cs))
    print(f'\n=== 1000-perm test [{defn.upper()}]  '
          f'N={len(ids[defn])}  fixed_C={fixed_C:.4f} ===')
    t0 = time.time()
    perms[defn] = permutation_test_no_filter(
        X[defn], y[defn], observed_auroc=res.auroc,
        fixed_C=fixed_C, n_perm=N_PERMUTATIONS_LONG,
        seed=RANDOM_SEED, progress_every=200,
    )
    p = perms[defn]
    print(f'  done {time.time()-t0:.0f}s  '
          f'p={p["p_value"]:.4f}  '
          f'null_mean={np.mean(p["perm_aurocs"]):.4f}  '
          f'null_max={np.max(p["perm_aurocs"]):.4f}')



=== 1000-perm test [PRIMARY]  N=20  fixed_C=0.3162 ===
    perm 200/1000  (2s)
    perm 400/1000  (4s)
    perm 600/1000  (6s)
    perm 800/1000  (9s)
    perm 1000/1000  (11s)
  done 11s  p=0.0000  null_mean=0.4663  null_max=0.9167

=== 1000-perm test [PRAGMATIC]  N=31  fixed_C=0.3162 ===
    perm 200/1000  (4s)
    perm 400/1000  (7s)
    perm 600/1000  (11s)
    perm 800/1000  (15s)
    perm 1000/1000  (19s)
  done 19s  p=0.0100  null_mean=0.4791  null_max=0.9359


## 6 — Analysis 2: Known-sex-only pragmatic sensitivity

Drop the four samples with Unknown sex annotation (ACH-001813, ACH-001817, ACH-002779, ACH-003573) from the pragmatic cohort. Remaining: N=27 (pos=17, neg=10). Re-run D3 LOOCV + 1000-perm + full refit to compare AUROC and gene coefficients against the full pragmatic (N=31).

In [7]:
print(f'=== D3 LOOCV [pragmatic known-sex only]  N={len(ids_ks)} ===')
print(f'  pos={y_ks.sum()}  neg={(y_ks==0).sum()}  dropped={sorted(UNKNOWN_SEX_IDS)}')
t0 = time.time()
loocv_ks, fold_coefs_ks = run_loocv_no_filter_with_coefs(X_ks, y_ks, ids_ks)
fixed_C_ks = float(np.median(loocv_ks.selected_Cs))
print(f'  {time.time()-t0:.1f}s  AUROC={loocv_ks.auroc:.4f}  '
      f'median_C={fixed_C_ks:.4f}')

metrics_ks = evaluate(loocv_ks.y_true, loocv_ks.y_prob)
cis_ks = bootstrap_ci(loocv_ks.y_true, loocv_ks.y_prob,
                       n_boot=N_BOOTSTRAP, seed=RANDOM_SEED)
m_ks, ci_ks = metrics_ks, cis_ks
print(f'  AUROC={m_ks["auroc"]:.4f} '
      f'[{ci_ks["auroc"][0]:.4f},{ci_ks["auroc"][1]:.4f}]  '
      f'AUPRC={m_ks["auprc"]:.4f}  '
      f'sens@90spec={m_ks["sensitivity_at_90_spec"]:.4f}')

print(f'\n=== 1000-perm test [pragmatic known-sex only]  fixed_C={fixed_C_ks:.4f} ===')
t0 = time.time()
perms_ks = permutation_test_no_filter(
    X_ks, y_ks, observed_auroc=loocv_ks.auroc,
    fixed_C=fixed_C_ks, n_perm=N_PERMUTATIONS_LONG,
    seed=RANDOM_SEED, progress_every=200,
)
pk = perms_ks
print(f'  done {time.time()-t0:.0f}s  '
      f'p={pk["p_value"]:.4f}  '
      f'null_mean={np.mean(pk["perm_aurocs"]):.4f}')

# Full refit on known-sex pragmatic for coefficient comparison
sc_ks_full = StandardScaler()
X_ks_s = sc_ks_full.fit_transform(X_ks)
clf_ks_full = build_estimator(C=fixed_C_ks, random_state=RANDOM_SEED)
clf_ks_full.fit(X_ks_s, y_ks)
coef_ks_full = clf_ks_full.coef_.ravel()

# Full refit on pragmatic (for comparison)
fixed_C_prag = float(np.median(loocv_res['pragmatic'].selected_Cs))
sc_prag_full = StandardScaler()
X_prag_s = sc_prag_full.fit_transform(X['pragmatic'])
clf_prag_full = build_estimator(C=fixed_C_prag, random_state=RANDOM_SEED)
clf_prag_full.fit(X_prag_s, y['pragmatic'])
coef_prag_full = clf_prag_full.coef_.ravel()

print('\n--- Coefficient comparison: pragmatic (N=31) vs known-sex (N=27) ---')
coef_compare = pd.DataFrame({
    'gene': curated_syms,
    'coef_prag_N31': coef_prag_full,
    'coef_ks_N27': coef_ks_full,
})
coef_compare['delta'] = coef_compare['coef_ks_N27'] - coef_compare['coef_prag_N31']
coef_compare['abs_prag'] = coef_compare['coef_prag_N31'].abs()
coef_compare = coef_compare.sort_values('abs_prag', ascending=False).reset_index(drop=True)
coef_compare.insert(0, 'rank_N31', range(1, len(coef_compare) + 1))
print(coef_compare[['rank_N31','gene','coef_prag_N31','coef_ks_N27','delta']].to_string(index=False))


=== D3 LOOCV [pragmatic known-sex only]  N=27 ===
  pos=17  neg=10  dropped=['ACH-001813', 'ACH-001817', 'ACH-002779', 'ACH-003573']
  2.6s  AUROC=0.8765  median_C=0.3162
  AUROC=0.8765 [0.7176,0.9835]  AUPRC=0.9369  sens@90spec=0.8235

=== 1000-perm test [pragmatic known-sex only]  fixed_C=0.3162 ===
    perm 200/1000  (3s)
    perm 400/1000  (6s)
    perm 600/1000  (9s)
    perm 800/1000  (12s)
    perm 1000/1000  (15s)
  done 15s  p=0.0030  null_mean=0.4719

--- Coefficient comparison: pragmatic (N=31) vs known-sex (N=27) ---
 rank_N31     gene  coef_prag_N31  coef_ks_N27     delta
        1    FANCM       0.767177     0.632474 -0.134702
        2     TERT      -0.754628    -0.882270 -0.127642
        3     TP53      -0.336636    -0.100740  0.235896
        4      NBN       0.237621     0.269259  0.031638
        5      WRN      -0.219992    -0.300306 -0.080314
        6     ATRX      -0.124507    -0.169728 -0.045221
        7    BRCA2       0.116268     0.000000 -0.116268
        8

## 7 — Analysis 3: Bootstrap coefficient stability

200 bootstrap resamples of the training set (with replacement), each refit at the fold-median C from the LOOCV. Reports per-gene non-zero frequency and classifies each gene as **stable** (>80%), **borderline** (50–80%), or **unstable** (<50%). Run for primary, pragmatic (full, N=31), and pragmatic known-sex (N=27).

In [8]:
boot_freq = {}   # key -> (freq_array, n_valid)

# Primary
fixed_C_prim = float(np.median(loocv_res['primary'].selected_Cs))
print(f'=== Bootstrap stability [PRIMARY]  '
      f'N={len(ids["primary"])}  n_boot={N_BOOTSTRAP_COEF}  C={fixed_C_prim:.4f} ===')
t0 = time.time()
freq_prim, nv_prim = bootstrap_coef_stability(
    X['primary'], y['primary'], fixed_C=fixed_C_prim,
    n_boot=N_BOOTSTRAP_COEF, seed=RANDOM_SEED,
)
boot_freq['primary'] = (freq_prim, nv_prim)
print(f'  {time.time()-t0:.1f}s  valid_resamples={nv_prim}/{N_BOOTSTRAP_COEF}')
df_prim_boot = pd.DataFrame({
    'gene': curated_syms,
    'nonzero_freq': np.round(freq_prim, 4),
    'stability': classify_stability(freq_prim),
}).sort_values('nonzero_freq', ascending=False).reset_index(drop=True)
print(df_prim_boot.to_string(index=False))

# Pragmatic (full, N=31)
print(f'\n=== Bootstrap stability [PRAGMATIC N=31]  '
      f'n_boot={N_BOOTSTRAP_COEF}  C={fixed_C_prag:.4f} ===')
t0 = time.time()
freq_prag, nv_prag = bootstrap_coef_stability(
    X['pragmatic'], y['pragmatic'], fixed_C=fixed_C_prag,
    n_boot=N_BOOTSTRAP_COEF, seed=RANDOM_SEED,
)
boot_freq['pragmatic'] = (freq_prag, nv_prag)
print(f'  {time.time()-t0:.1f}s  valid_resamples={nv_prag}/{N_BOOTSTRAP_COEF}')
df_prag_boot = pd.DataFrame({
    'gene': curated_syms,
    'nonzero_freq': np.round(freq_prag, 4),
    'stability': classify_stability(freq_prag),
}).sort_values('nonzero_freq', ascending=False).reset_index(drop=True)
print(df_prag_boot.to_string(index=False))

# Pragmatic known-sex (N=27)
print(f'\n=== Bootstrap stability [PRAGMATIC known-sex N=27]  '
      f'n_boot={N_BOOTSTRAP_COEF}  C={fixed_C_ks:.4f} ===')
t0 = time.time()
freq_ks_boot, nv_ks = bootstrap_coef_stability(
    X_ks, y_ks, fixed_C=fixed_C_ks,
    n_boot=N_BOOTSTRAP_COEF, seed=RANDOM_SEED,
)
boot_freq['pragmatic_ks'] = (freq_ks_boot, nv_ks)
print(f'  {time.time()-t0:.1f}s  valid_resamples={nv_ks}/{N_BOOTSTRAP_COEF}')
df_ks_boot = pd.DataFrame({
    'gene': curated_syms,
    'nonzero_freq': np.round(freq_ks_boot, 4),
    'stability': classify_stability(freq_ks_boot),
}).sort_values('nonzero_freq', ascending=False).reset_index(drop=True)
print(df_ks_boot.to_string(index=False))


=== Bootstrap stability [PRIMARY]  N=20  n_boot=200  C=0.3162 ===
  0.1s  valid_resamples=200/200
    gene  nonzero_freq  stability
    TERT         1.000     stable
   FANCM         1.000     stable
   BRCA2         0.810     stable
     NBN         0.625 borderline
     WRN         0.570 borderline
    TP53         0.445   unstable
RAD51AP1         0.280   unstable
    ATRX         0.235   unstable
   BRCA1         0.210   unstable
  WRAP53         0.170   unstable
    DAXX         0.165   unstable
SMARCAL1         0.160   unstable
     BLM         0.120   unstable
    SLX4         0.110   unstable
   MUS81         0.110   unstable
   RAD52         0.080   unstable
   RAD50         0.080   unstable
    GEN1         0.075   unstable
  FANCD2         0.030   unstable

=== Bootstrap stability [PRAGMATIC N=31]  n_boot=200  C=0.3162 ===
  0.2s  valid_resamples=200/200
    gene  nonzero_freq  stability
    TERT         1.000     stable
   FANCM         1.000     stable
    TP53         0.9

/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## 8 — Analysis 4: LOOCV fold coefficient frequency

From the LOOCV runs above, `fold_coefs[i]` is the coefficient vector from the fold that held out sample i. The per-gene non-zero frequency = fraction of N folds where |coef| > 1e-10. Complements bootstrap stability (which captures sampling variability; this captures leave-one-out variability).

In [9]:
fold_freq = {}  # key -> ndarray (n_feat,) — fraction of folds with nonzero coef

for defn in DEFINITIONS:
    fc = fold_coefs[defn]
    n_folds = fc.shape[0]
    nz = (np.abs(fc) > NONZERO_THRESH)
    freq = nz.mean(axis=0)
    fold_freq[defn] = freq
    df = pd.DataFrame({
        'gene': curated_syms,
        'fold_nonzero_freq': np.round(freq, 4),
        'n_folds_nonzero': nz.sum(axis=0),
        'n_folds_total': n_folds,
    }).sort_values('fold_nonzero_freq', ascending=False).reset_index(drop=True)
    print(f'\n[{defn}] LOOCV fold coefficient frequency  (N_folds={n_folds})')
    print(df.to_string(index=False))

# Known-sex pragmatic
fc_ks = fold_coefs_ks
nz_ks = (np.abs(fc_ks) > NONZERO_THRESH)
freq_ks_fold = nz_ks.mean(axis=0)
fold_freq['pragmatic_ks'] = freq_ks_fold
df_ks_fold = pd.DataFrame({
    'gene': curated_syms,
    'fold_nonzero_freq': np.round(freq_ks_fold, 4),
    'n_folds_nonzero': nz_ks.sum(axis=0),
    'n_folds_total': fc_ks.shape[0],
}).sort_values('fold_nonzero_freq', ascending=False).reset_index(drop=True)
print(f'\n[pragmatic known-sex] LOOCV fold coefficient frequency  (N_folds={fc_ks.shape[0]})')
print(df_ks_fold.to_string(index=False))



[primary] LOOCV fold coefficient frequency  (N_folds=20)
    gene  fold_nonzero_freq  n_folds_nonzero  n_folds_total
    TERT               1.00               20             20
   FANCM               1.00               20             20
   BRCA2               0.95               19             20
     WRN               0.90               18             20
     NBN               0.90               18             20
    TP53               0.50               10             20
    ATRX               0.10                2             20
   RAD52               0.00                0             20
    GEN1               0.00                0             20
    SLX4               0.00                0             20
   BRCA1               0.00                0             20
  FANCD2               0.00                0             20
  WRAP53               0.00                0             20
RAD51AP1               0.00                0             20
    DAXX               0.00               

## 9 — Results summary

In [10]:
# ── AUROC comparison table ───────────────────────────────────────────────────
def _f(v, fmt='.4f'):
    return 'N/A' if v is None else format(float(v), fmt)

def _ci(ci_dict, key):
    lo, hi = ci_dict.get(key, (float('nan'), float('nan')))
    return f'[{lo:.4f},{hi:.4f}]' if lo == lo else 'N/A'

auroc_rows = []
for defn in DEFINITIONS:
    m, ci = metrics[defn], cis[defn]
    p1k = perms[defn]['p_value']
    auroc_rows.append({'Cohort': defn, 'N': len(ids[defn]),
                       'AUROC': _f(m['auroc']),
                       '95% CI (boot)': _ci(ci, 'auroc'),
                       'AUPRC': _f(m['auprc']),
                       'Sens@90Spec': _f(m['sensitivity_at_90_spec']),
                       'Perm p (1000)': _f(p1k, '.4f')})
auroc_rows.append({'Cohort': 'pragmatic_ks (N=27)', 'N': len(ids_ks),
                   'AUROC': _f(m_ks['auroc']),
                   '95% CI (boot)': _ci(cis_ks, 'auroc'),
                   'AUPRC': _f(m_ks['auprc']),
                   'Sens@90Spec': _f(m_ks['sensitivity_at_90_spec']),
                   'Perm p (1000)': _f(perms_ks['p_value'], '.4f')})

auroc_df = pd.DataFrame(auroc_rows)
print('=== AUROC Comparison (with 1000-permutation p-values) ===')
print(auroc_df.to_string(index=False))

# ── Stability cross-table ─────────────────────────────────────────────────────
stability_rows = []
cohort_keys = [('primary', 'primary', freq_prim, fold_freq['primary']),
               ('pragmatic', 'pragmatic', freq_prag, fold_freq['pragmatic']),
               ('pragmatic_ks', 'pragmatic_ks', freq_ks_boot, fold_freq['pragmatic_ks'])]

for key, label, boot_f, fold_f in cohort_keys:
    stab = classify_stability(boot_f)
    for j, sym in enumerate(curated_syms):
        stability_rows.append({
            'cohort': label, 'gene': sym,
            'boot_nonzero_freq': round(float(boot_f[j]), 4),
            'fold_nonzero_freq': round(float(fold_f[j]), 4),
            'stability': stab[j],
        })

stability_df = pd.DataFrame(stability_rows)

print('\n=== Bootstrap stability — PRIMARY ===')
p_stab = stability_df[stability_df['cohort']=='primary'].sort_values(
    'boot_nonzero_freq', ascending=False)
print(p_stab[['gene','boot_nonzero_freq','fold_nonzero_freq','stability']].to_string(index=False))

print('\n=== Bootstrap stability — PRAGMATIC (N=31) ===')
g_stab = stability_df[stability_df['cohort']=='pragmatic'].sort_values(
    'boot_nonzero_freq', ascending=False)
print(g_stab[['gene','boot_nonzero_freq','fold_nonzero_freq','stability']].to_string(index=False))

# Genes that are stable in BOTH primary and pragmatic
p_stable = set(p_stab[p_stab['stability']=='stable']['gene'])
g_stable = set(g_stab[g_stab['stability']=='stable']['gene'])
both_stable = p_stable & g_stable
print(f'\nGenes stable in both primary and pragmatic ({len(both_stable)}): '
      f'{sorted(both_stable)}')


=== AUROC Comparison (with 1000-permutation p-values) ===
             Cohort  N  AUROC   95% CI (boot)  AUPRC Sens@90Spec Perm p (1000)
            primary 20 0.9479 [0.8125,1.0000] 0.9755      0.9167        0.0000
          pragmatic 31 0.8291 [0.6681,0.9625] 0.9166      0.7778        0.0100
pragmatic_ks (N=27) 27 0.8765 [0.7176,0.9835] 0.9369      0.8235        0.0030

=== Bootstrap stability — PRIMARY ===
    gene  boot_nonzero_freq  fold_nonzero_freq  stability
    TERT              1.000               1.00     stable
   FANCM              1.000               1.00     stable
   BRCA2              0.810               0.95     stable
     NBN              0.625               0.90 borderline
     WRN              0.570               0.90 borderline
    TP53              0.445               0.50   unstable
RAD51AP1              0.280               0.00   unstable
    ATRX              0.235               0.10   unstable
   BRCA1              0.210               0.00   unstable
  WRAP5

## 10 — Write outputs and report

In [11]:
output_paths = {}

# ── Prediction CSVs ─────────────────────────────────────────────────────────
def _write_pred(res, name):
    df = pd.DataFrame({'sample_id': res.sample_ids,
                       'true_label': res.y_true,
                       'predicted_prob': res.y_prob})
    p = PROC / name
    df.to_csv(p, index=False)
    output_paths[name] = p
    return p

_write_pred(loocv_res['primary'], 'aim2_curated_primary_rob_predictions.csv')
_write_pred(loocv_res['pragmatic'], 'aim2_curated_pragmatic_rob_predictions.csv')
_write_pred(loocv_ks, 'aim2_curated_pragmatic_ks_predictions.csv')

# ── Stability CSV ─────────────────────────────────────────────────────────────
stab_path = PROC / 'aim2_curated_stability.csv'
stability_df.to_csv(stab_path, index=False)
output_paths['aim2_curated_stability.csv'] = stab_path

for name in sorted(output_paths):
    p = output_paths[name]
    print(f'  wrote {name}  ({p.stat().st_size:,} bytes)')

# ── Markdown report ──────────────────────────────────────────────────────────
L = []
L.append('# Notebook 13 — Aim 2 D3 Curated ALT: Robustness Analysis\n\n')
L.append(f'Run timestamp: `{RUN_TIMESTAMP}`\n\n')
L.append('Four extensions to the notebook 12 D3 (curated ALT-only) analysis. '
         'See notebook 12 for the baseline D3 results.\n\n')

L.append('## Configuration\n\n')
L.append(f'- Curated gene list: {len(curated_syms)}/{len(ALT_GENE_LIST)} found  '
         f'(not found: {not_found_curated})\n')
L.append(f'- 1000-perm test: fixed median C, precomputed fold scalers (same '
         f'results as re-running full pipeline per shuffle, faster).\n')
L.append(f'- Bootstrap stability: {N_BOOTSTRAP_COEF} resamples, '
         f'NONZERO_THRESH={NONZERO_THRESH}.\n')
L.append(f'- Random seed: {RANDOM_SEED}.  '
         f'CI bootstrap resamples: {N_BOOTSTRAP}.\n\n')

L.append('## Analysis 1: D3 with 1000-permutation test\n\n')
L.append('| Cohort | N | AUROC | 95% CI | AUPRC | Sens@90Spec | Perm p (1000) |\n')
L.append('|---|---|---|---|---|---|---|\n')
for r in auroc_rows:
    L.append(f"| {r['Cohort']} | {r['N']} | {r['AUROC']} | "
             f"{r['95% CI (boot)']} | {r['AUPRC']} | "
             f"{r['Sens@90Spec']} | {r['Perm p (1000)']} |\n")
L.append('\n')

L.append('## Analysis 2: Known-sex-only pragmatic sensitivity\n\n')
L.append(f'Dropped samples: {sorted(UNKNOWN_SEX_IDS)}\n\n')
L.append(f'N=31 AUROC={metrics["pragmatic"]["auroc"]:.4f}  '
         f'N=27 AUROC={m_ks["auroc"]:.4f}  '
         f'delta={m_ks["auroc"]-metrics["pragmatic"]["auroc"]:+.4f}\n\n')
L.append('### Coefficient comparison (full vs known-sex, sorted by |coef| N=31)\n\n')
L.append('| Rank_N31 | Gene | Coef N=31 | Coef N=27 | Delta |\n')
L.append('|---|---|---|---|---|\n')
for _, r in coef_compare.iterrows():
    L.append(f"| {r['rank_N31']} | {r['gene']} | "
             f"{r['coef_prag_N31']:+.6f} | "
             f"{r['coef_ks_N27']:+.6f} | "
             f"{r['delta']:+.6f} |\n")
L.append('\n')

L.append('## Analysis 3: Bootstrap coefficient stability\n\n')
L.append('| Cohort | N | stable (>80%) | borderline (50-80%) | unstable (<50%) |\n')
L.append('|---|---|---|---|---|\n')
for key, label, boot_f, _ in cohort_keys:
    stab = classify_stability(boot_f)
    n_stable = int((stab=='stable').sum())
    n_border = int((stab=='borderline').sum())
    n_unstable = int((stab=='unstable').sum())
    n_samp = len(ids[key]) if key in ids else len(ids_ks)
    L.append(f'| {label} | {n_samp} | {n_stable} genes | '
             f'{n_border} genes | {n_unstable} genes |\n')
L.append('\n')

L.append(f'Genes stable in both primary and pragmatic ({len(both_stable)}): '
         f'{sorted(both_stable)}\n\n')

# Per-gene bootstrap table (primary + pragmatic side-by-side)
L.append('### Per-gene bootstrap non-zero frequency\n\n')
L.append('| Gene | Primary boot_freq | Primary stability | '
         'Pragmatic boot_freq | Pragmatic stability |\n')
L.append('|---|---|---|---|---|\n')
p_map = {r['gene']: r for _, r in p_stab.iterrows()}
g_map = {r['gene']: r for _, r in g_stab.iterrows()}
for sym in curated_syms:
    pr = p_map.get(sym, {})
    gr = g_map.get(sym, {})
    L.append(f"| {sym} | {pr.get('boot_nonzero_freq','N/A')} | "
             f"{pr.get('stability','N/A')} | "
             f"{gr.get('boot_nonzero_freq','N/A')} | "
             f"{gr.get('stability','N/A')} |\n")
L.append('\n')

L.append('## Analysis 4: LOOCV fold coefficient frequency\n\n')
L.append('| Gene | Primary fold_freq | Pragmatic fold_freq | Pragmatic_ks fold_freq |\n')
L.append('|---|---|---|---|\n')
for j, sym in enumerate(curated_syms):
    L.append(f'| {sym} | '
             f'{fold_freq["primary"][j]:.4f} | '
             f'{fold_freq["pragmatic"][j]:.4f} | '
             f'{fold_freq["pragmatic_ks"][j]:.4f} |\n')
L.append('\n')

report_path = PROC / 'notebook_13_report.md'
report_path.write_text(''.join(L))
output_paths['notebook_13_report.md'] = report_path
print(f'\nReport: notebook_13_report.md  ({report_path.stat().st_size:,} bytes)')


  wrote aim2_curated_pragmatic_ks_predictions.csv  (909 bytes)
  wrote aim2_curated_pragmatic_rob_predictions.csv  (1,035 bytes)
  wrote aim2_curated_primary_rob_predictions.csv  (679 bytes)
  wrote aim2_curated_stability.csv  (2,124 bytes)

Report: notebook_13_report.md  (4,432 bytes)


## 11 — SHA-256 idempotence check

In [12]:
SHA_PATH = PROC / 'notebook_13_sha256.json'
HASHED = sorted(output_paths.keys())
current = {name: sha256_file(output_paths[name]) for name in HASHED}
for name in HASHED:
    print(f'  {name}: {current[name]}')

if SHA_PATH.exists():
    previous = json.loads(SHA_PATH.read_text())
    mismatch = {
        k: (previous.get(k), current[k])
        for k in HASHED if previous.get(k) != current[k]
    }
    if mismatch:
        print('\nIDEMPOTENCE FAILED — outputs differ from recorded sha256:')
        for k, (was, now) in mismatch.items():
            print(f'  {k}: was {was} now {now}')
        raise AssertionError(f'Idempotence FAILED for: {sorted(mismatch)}')
    print('\nIdempotence check PASSED: all output files byte-identical to recorded sha256.')
else:
    print('\nFirst run: recording sha256 baseline (re-run to verify byte-identity).')

SHA_PATH.write_text(json.dumps(current, indent=2, sort_keys=True) + '\n')
print(f'sha256 manifest: {SHA_PATH.relative_to(ROOT)}')


  aim2_curated_pragmatic_ks_predictions.csv: fe2eafe5d9fd47cf03641eef0691daad1741ede7576970747d8904a19f19abcf
  aim2_curated_pragmatic_rob_predictions.csv: b1fcdc215ad0817484897d7fc1a7e52bc73c7aac527622408ca48837ea8bf30d
  aim2_curated_primary_rob_predictions.csv: 62c18e5fe467891f131345ba5884de9e4a1f64f7ffd709a0763569c7f2173e76
  aim2_curated_stability.csv: 9c44991bd494e02726a7f2246fbbc9ff34ae4b8ebc494e2f559b33171c236e80
  notebook_13_report.md: 0b217ea0055642a8a909e40c7cc6a2e7ccdc219d3799b2ca7db4f6f5783371b7

First run: recording sha256 baseline (re-run to verify byte-identity).
sha256 manifest: data/processed/notebook_13_sha256.json
